# 10年定着予測 - 入社年の外挿問題と全特徴量の外挿監査（41_）

**背景**: `literature/01_early_turnover_literature_review.md`（文献レビュー、2026-08-12）で、
小林徹(2016)が「学卒時の有効求人倍率が3年内離職率の約3%ptを説明する」と定量化しているのを受けて
本データのコホート変数を点検したところ、**特徴量設計の不備**が見つかった。

## 見つかった問題

```
Train: n=2761  入社日 2011-04-01 〜 2014-03-01
Test : n=2502  入社日 2014-04-01 〜 2017-03-01   ← 入社年で完全に分離
```

にもかかわらず `28_`〜`37_` は **`入社年` を441特徴量の1つとして渡している**
（`_feature_cols` が除外するのは `入社日` だけで、`入社年`・`入社月`・`入社四半期` は
persona列としてそのまま残る）。

| 入社年 | 学習80% | 検証20% | Test |
|---|---|---|---|
| 2011 | 836 | 0 | 0 |
| 2012 | 971 | 0 | 0 |
| 2013 | 401 | 493 | 0 |
| 2014 | 0 | **60** | 829 |
| 2015 | 0 | 0 | 855 |
| 2016 | 0 | 0 | 809 |
| 2017 | 0 | 0 | 9 |

CatBoostは `入社年` を数値扱いするので、`入社年 > 2013.5` の分岐が1本でもあれば
**Test 2,502名全員が、学習側60名の葉に落ちる**。

## ローカルで済ませた検証（第10節で再現する）

| 問い | 結果 |
|---|---|
| 学習範囲内(2011-2013)で `入社年` に信号はあるか | **無い**。振れ幅3.2%pt、カイ二乗 p=0.345 |
| 「入社年≥2014」の学習側の証拠の強さ | **60名のみ**。定着率0.6500、95%CI [0.5245, 0.7614] は ≤2013 の 0.5628 を含む（Fisher p=0.190） |
| D3提出のTest予測を入社年で層別 | 0.5860 / 0.6058 / 0.5694（振れ幅0.038）。ただし**他特徴量の分布差も混ざるため単独寄与は分離できない** |

**本ノートブックの目的は、この3つ目を分離すること。**
`入社年` 以外すべて同一・同一シードの2モデルを学習すれば、Test予測の差がそのまま `入社年` の寄与になる。

## 実行構成

**ベースラインは第12節手前の `BASE` 定数1つで切り替えられる**（`40_` からグループ定義を移植済み）。

| `BASE` | 列数 | 対応する 40_ の構成 |
|---|---|---|
| `"full441"` （既定） | 441 | `R0_ref`（= `37_` D3、Public 0.522659） |
| `"no_tfidf396"` | 396 | `R1_no_tfidf` |
| `"lean113"` | 113 | `R6_lean` |

`40_` の Public 結果を見て、勝った構成に合わせてここを変えてから実行する。
`入社年` は `persona` グループにあるため**どのベースラインにも含まれており**、
41_ の問い自体はベースラインに依存しない。

ハイパーパラメータは `A_PARAMS` 固定、反復数は 560 固定（`D3` と同一）。
したがって各構成は現最良 `D3`（Public 0.522659）と**特徴量だけが違う**。

| config | 特徴量 | 位置づけ |
|---|---|---|
| `X0_ref` | 全441列 | **参照**。`D3` の再現。ここが再現しなければ以降は無効 |
| `X1_no_hireyear` | −`入社年`（1列だけ） | **本命**。R0からの変更点が1列なので寄与を完全に分離できる |
| `X2_no_all_hiretime` | −`入社年`・`入社月`・`入社四半期` | 循環変数まで落とす保険。`入社月`は外挿問題を持たないので本来不要 |
| `X3_no_extrapolating` | 第11節の監査で検出された外挿列をすべて除外 | **事前登録したルールで自動決定**。検出結果が `入社年` だけなら X1 と同一になるので自動スキップ |

## 判定方法（事前登録）

`39_` の教訓により、**採否は Public でのみ決める**。
検証スコア（生存者535名）は分解能±0.011しかなく、`入社年` の期待効果はそれ以下なので、
**検証スコアでは原理的に判定できない**。ここでは足切り（`X0`から+0.02以上の悪化）にのみ使う。

## 期待値について（過大評価しないこと）

CatBoostは depth=4・強い正則化で動いており、無信号の列に大きく分岐している可能性は高くない。
**期待改善幅は 0.000〜0.003 程度**で、これは得点を伸ばす施策というより
**明確な設計上の不備を潰す**位置づけである。
第15節で `入社年` の特徴量重要度と予測差を測れば、効果の上限が事前にわかる。

## 実行環境

Google Colab Pro の **CPUハイメモリ**ランタイム。
CatBoost の学習回数は最大64回（560反復・CPU）で、想定実行時間は **30〜60分**。

> ⚠️ **ローカルMacで先行実行しないこと。** `27_` で発生したチェックポイントのGoogle Drive同期事故を
> 避けるため、本ノートブックはColabで直接実行する。やり直したい場合は `RESET_CHECKPOINT = True` にする。


In [29]:
!pip install -q catboost optuna

In [30]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [31]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [33]:
SCRIPT_NAME = "41_hire_year_extrapolation"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-12 00:18:26] [INFO] === [41_hire_year_extrapolation] 実験開始 ===


INFO:41_hire_year_extrapolation:=== [41_hire_year_extrapolation] 実験開始 ===


[2026-08-12 00:18:26] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


INFO:41_hire_year_extrapolation:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260812


[2026-08-12 00:18:26] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/41_hire_year_extrapolation_checkpoint.csv


INFO:41_hire_year_extrapolation:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/41_hire_year_extrapolation_checkpoint.csv


[2026-08-12 00:18:26] [INFO] チェックポイントは未作成（新規実行）


INFO:41_hire_year_extrapolation:チェックポイントは未作成（新規実行）


In [34]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-12 00:18:27] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:41_hire_year_extrapolation:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-12 00:18:27] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:41_hire_year_extrapolation:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-12 00:18:27] [INFO] 定着率: 0.5647


INFO:41_hire_year_extrapolation:定着率: 0.5647


[2026-08-12 00:18:27] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:41_hire_year_extrapolation:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [35]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-12 00:18:27] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:41_hire_year_extrapolation:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-12 00:18:27] [INFO] Test  早期退職者: 0名 / 2502名


INFO:41_hire_year_extrapolation:Test  早期退職者: 0名 / 2502名


[2026-08-12 00:18:27] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:41_hire_year_extrapolation:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-12 00:18:27] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:41_hire_year_extrapolation:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [36]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [37]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-12 00:18:28] [INFO] ------------------------------------------------------------


INFO:41_hire_year_extrapolation:------------------------------------------------------------


[2026-08-12 00:18:28] [INFO] split非依存の基本特徴量を生成中...


INFO:41_hire_year_extrapolation:split非依存の基本特徴量を生成中...


[2026-08-12 00:18:28] [INFO] ------------------------------------------------------------


INFO:41_hire_year_extrapolation:------------------------------------------------------------


[2026-08-12 00:26:44] [INFO] split非依存の基本特徴量生成完了


INFO:41_hire_year_extrapolation:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [38]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-12 00:26:44] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:41_hire_year_extrapolation:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-12 00:26:46] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:41_hire_year_extrapolation:入社時メモ: SVD累積寄与率=0.760


[2026-08-12 00:26:52] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:41_hire_year_extrapolation:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-12 00:26:54] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:41_hire_year_extrapolation:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-12 00:26:54] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:41_hire_year_extrapolation:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [39]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-12 00:26:54] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:41_hire_year_extrapolation:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-12 00:30:01] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:41_hire_year_extrapolation:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [40]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-12 00:30:02] [INFO] Persona単位の基本特徴量を生成中...


INFO:41_hire_year_extrapolation:Persona単位の基本特徴量を生成中...


[2026-08-12 00:30:02] [INFO] Persona単位の基本特徴量処理完了


INFO:41_hire_year_extrapolation:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [41]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-12 00:30:02] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:41_hire_year_extrapolation:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-12 00:30:02] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:41_hire_year_extrapolation:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-12 00:30:02] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:41_hire_year_extrapolation:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [42]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [43]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [44]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [45]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-12 00:30:03] [INFO] ============================================================


INFO:41_hire_year_extrapolation:============================================================


[2026-08-12 00:30:03] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:41_hire_year_extrapolation:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-12 00:30:03] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:41_hire_year_extrapolation:B/C: train=2208, val=535（生存者のみ）


[2026-08-12 00:30:04] [INFO] D: train=2761（全件）, val=0（空）


INFO:41_hire_year_extrapolation:D: train=2761（全件）, val=0（空）


[2026-08-12 00:30:04] [INFO] 特徴量数: 441


INFO:41_hire_year_extrapolation:特徴量数: 441


## 10. ローカル検証の再現（学習不要）

In [46]:
# ============================================================
# 第10節: ローカルで行った検証の再現（学習不要）
#   文献レビュー第3.1節の問1〜問3をノートブック上で再現し、
#   結果を実行ログに残す。
# ============================================================

from scipy import stats as _st

_tr = train_persona.copy()
_te = test_persona.copy()
_tr["_year"] = pd.to_datetime(_tr["入社日"]).dt.year
_te["_year"] = pd.to_datetime(_te["入社日"]).dt.year
_y = train_persona.set_index(ID_COL)[TARGET_COL]

print("=" * 70)
print("前提: Train と Test の入社日レンジ")
print("=" * 70)
print(f"  Train: n={len(_tr)}  {pd.to_datetime(_tr['入社日']).min().date()} 〜 {pd.to_datetime(_tr['入社日']).max().date()}")
print(f"  Test : n={len(_te)}  {pd.to_datetime(_te['入社日']).min().date()} 〜 {pd.to_datetime(_te['入社日']).max().date()}")

_dist = pd.DataFrame({
    "Train": _tr["_year"].value_counts(),
    "Test": _te["_year"].value_counts(),
}).fillna(0).astype(int).sort_index()
_dist["Train定着率"] = _tr.groupby("_year")[TARGET_COL].mean().round(4)
print()
print(_dist.to_string())

_overlap = set(_tr["_year"]) & set(_te["_year"])
print()
print(f"  Train/Test 共通の入社年: {sorted(_overlap)}")
print(f"  Testのうち学習範囲外(>{_tr['_year'].max()})の人数: "
      f"{(_te['_year'] > _tr['_year'].max()).sum()} / {len(_te)}")

print()
print("=" * 70)
print("問1: 学習範囲内(2011-2013)で 入社年 に信号はあるか")
print("=" * 70)
_sub = _tr[_tr["_year"] <= 2013]
_ct = pd.crosstab(_sub["_year"], _sub[TARGET_COL])
_chi2, _p1, _dof, _ = _st.chi2_contingency(_ct)
_rate = _sub.groupby("_year")[TARGET_COL].mean()
print(_ct.to_string())
print(f"\n  定着率: " + " / ".join(f"{k}={v:.4f}" for k, v in _rate.items()))
print(f"  振れ幅: {100 * (_rate.max() - _rate.min()):.1f}%pt")
print(f"  カイ二乗検定: chi2={_chi2:.3f}, dof={_dof}, p={_p1:.4f}")
print(f"  → {'有意' if _p1 < 0.05 else '有意でない'}（本プロジェクトの採用基準は20%pt以上・p<1e-20）")

print()
print("=" * 70)
print("問2: 「入社年≥2014」について学習データが持つ証拠の強さ")
print("=" * 70)
_a = _tr[_tr["_year"] >= 2014][TARGET_COL]
_b = _tr[_tr["_year"] <= 2013][TARGET_COL]
_lo, _hi = _st.beta.interval(0.95, _a.sum() + 0.5, len(_a) - _a.sum() + 0.5)
_, _p2 = _st.fisher_exact([[_a.sum(), len(_a) - _a.sum()], [_b.sum(), len(_b) - _b.sum()]])
print(f"  入社年≥2014: n={len(_a)}, 定着率 {_a.mean():.4f}, 95%CI [{_lo:.4f}, {_hi:.4f}]")
print(f"  入社年≤2013: n={len(_b)}, 定着率 {_b.mean():.4f}")
print(f"  差 {100 * (_a.mean() - _b.mean()):+.1f}%pt, Fisher正確検定 p={_p2:.4f}")
print(f"  → 95%CIは ≤2013 の {_b.mean():.4f} を"
      f"{'含む（区別できない）' if _lo <= _b.mean() <= _hi else '含まない'}")

print()
print("=" * 70)
print("問3: 既存の提出ファイル(D3)のTest予測を入社年で層別")
print("=" * 70)
_d3 = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_37_full_train_seed_averaging_D3_Aparams_full_x125.csv"))
if _d3:
    _sub_df = pd.read_csv(_d3[-1], header=None, names=[ID_COL, "pred"])
    _m = _te[[ID_COL, "_year"]].merge(_sub_df, on=ID_COL)
    assert len(_m) == len(_te), "社員IDが一致しない"
    print(f"  ファイル: {_d3[-1].name}（予測平均 {_m['pred'].mean():.4f}）")
    print(_m.groupby("_year")["pred"].agg(["size", "mean", "std"]).round(4).to_string())
    _rng = _m.groupby("_year")["pred"].mean()
    print(f"  → 入社年間の予測平均の振れ幅: {_rng.max() - _rng.min():.4f}")
    print("  ※ これは他特徴量の分布差も含むため、入社年単独の寄与ではない。")
    print("     第15節で X0 と X1 の予測差として分離する。")
else:
    print("  D3の提出ファイルが見つからなかった（スキップ）")

del _tr, _te, _sub, _ct, _a, _b


前提: Train と Test の入社日レンジ
  Train: n=2761  2011-04-01 〜 2014-03-01
  Test : n=2502  2014-04-01 〜 2017-03-01

       Train  Test  Train定着率
_year                       
2011     836     0    0.5682
2012     971     0    0.5448
2013     894     0    0.5772
2014      60   829    0.6500
2015       0   855       NaN
2016       0   809       NaN
2017       0     9       NaN

  Train/Test 共通の入社年: [2014]
  Testのうち学習範囲外(>2014)の人数: 1673 / 2502

問1: 学習範囲内(2011-2013)で 入社年 に信号はあるか
10年定着ラベル    0    1
_year             
2011      361  475
2012      442  529
2013      378  516

  定着率: 2011=0.5682 / 2012=0.5448 / 2013=0.5772
  振れ幅: 3.2%pt
  カイ二乗検定: chi2=2.128, dof=2, p=0.3450
  → 有意でない（本プロジェクトの採用基準は20%pt以上・p<1e-20）

問2: 「入社年≥2014」について学習データが持つ証拠の強さ
  入社年≥2014: n=60, 定着率 0.6500, 95%CI [0.5245, 0.7614]
  入社年≤2013: n=2701, 定着率 0.5628
  差 +8.7%pt, Fisher正確検定 p=0.1901
  → 95%CIは ≤2013 の 0.5628 を含む（区別できない）

問3: 既存の提出ファイル(D3)のTest予測を入社年で層別
  ファイル: 20260811_37_full_train_seed_averaging_D3_Aparams_full_x125.csv（予測

## 11. 全特徴量の外挿監査

In [47]:
# ============================================================
# 第11節: 全特徴量の外挿監査
#   入社年と同じ問題（Testが学習データのレンジ外に出る）を持つ列が
#   他にもないかを、441列すべてについて機械的に調べる。
#   学習は不要。
# ============================================================

# 変数名を AUDIT_FEATS にしているのは、11b節で 40_ から移植したセルが
# 同名のグローバルを set として再定義するため（名前の衝突を避ける）。
AUDIT_FEATS = _feature_cols(ag_train_80)
print(f"監査対象: {len(AUDIT_FEATS)} 列")

# 学習に実際に使われるのは「先頭80%」または「全件」。両方について見る。
_train_frames = {"学習80%": ag_train_80b, "全件学習": ag_full}

audit_rows = []
for col in AUDIT_FEATS:
    te_col = test_features_full[col]
    is_num = pd.api.types.is_numeric_dtype(ag_full[col]) and pd.api.types.is_numeric_dtype(te_col)
    row = {"feature": col, "dtype": "numeric" if is_num else "categorical"}

    for label, frm in _train_frames.items():
        tr_col = frm[col]
        if is_num:
            lo, hi = tr_col.min(), tr_col.max()
            if pd.isna(lo) or pd.isna(hi):
                frac = np.nan
            else:
                valid = te_col.notna()
                frac = float(((te_col < lo) | (te_col > hi))[valid].mean()) if valid.any() else np.nan
        else:
            seen = set(tr_col.dropna().unique())
            valid = te_col.notna()
            frac = float((~te_col[valid].isin(seen)).mean()) if valid.any() else np.nan
        row[label] = frac
    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)
audit["最大外挿率"] = audit[["学習80%", "全件学習"]].max(axis=1)
audit = audit.sort_values("最大外挿率", ascending=False)

# --- 事前登録した判定ルール ---
# Testの過半数が学習データのレンジ外（またはカテゴリ未出現）に落ちる列を「外挿列」とする。
# 閾値0.5は結果を見てから決めたものではなく、
# 「大半のTestが外挿域に落ちるなら、その列は予測に寄与できない」という理屈から事前に定めた。
EXTRAPOLATION_THRESHOLD = 0.5
EXTRAPOLATING_COLS = audit.loc[audit["最大外挿率"] >= EXTRAPOLATION_THRESHOLD, "feature"].tolist()

print()
print("外挿率の上位20列（Testのうち学習データのレンジ外／未出現カテゴリに落ちる割合）")
print("-" * 78)
print(audit.head(20).to_string(index=False))
print("-" * 78)
print(f"閾値 {EXTRAPOLATION_THRESHOLD} 以上の列: {len(EXTRAPOLATING_COLS)}件")
for c in EXTRAPOLATING_COLS:
    print(f"  - {c}")

assert "入社年" in EXTRAPOLATING_COLS, \
    "入社年が検出されない。第10節の前提と矛盾するので監査ロジックを疑うこと"
print()
print("✅ 入社年は監査で検出された（監査ロジックが機能している）")

audit.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_extrapolation_audit.csv", index=False)
logger.info(f"外挿監査を保存: {SCRIPT_NAME}_extrapolation_audit.csv")
logger.info(f"外挿列({EXTRAPOLATION_THRESHOLD}以上): {EXTRAPOLATING_COLS}")


監査対象: 441 列

外挿率の上位20列（Testのうち学習データのレンジ外／未出現カテゴリに落ちる割合）
------------------------------------------------------------------------------
                      feature   dtype    学習80%     全件学習    最大外挿率
                          入社年 numeric 1.000000 0.668665 1.000000
        360度評価_主体度_early_mean numeric 0.059524 0.023810 0.059524
  360度評価_主体度_late_early_ratio numeric 0.059524 0.035714 0.059524
      360度評価_共有貢献度_early_mean numeric 0.059524 0.000000 0.059524
  360度評価_親和度_late_early_ratio numeric 0.035714 0.035714 0.035714
360度評価_共有貢献度_late_early_ratio numeric 0.035714 0.035714 0.035714
  360度評価_主体度_late_minus_early numeric 0.035714 0.023810 0.035714
    360度評価者数_late_early_ratio numeric 0.023810 0.023810 0.023810
        360度評価_学習度_early_mean numeric 0.023810 0.000000 0.023810
        360度評価_親和度_early_mean numeric 0.023810 0.023810 0.023810
    360度評価者数_late_minus_early numeric 0.023810 0.023810 0.023810
  360度評価_学習度_late_early_ratio numeric 0.011905 0.011905 0.011905
360度評価_共有貢献度_late_mi

INFO:41_hire_year_extrapolation:外挿監査を保存: 41_hire_year_extrapolation_extrapolation_audit.csv


[2026-08-12 00:30:05] [INFO] 外挿列(0.5以上): ['入社年']


INFO:41_hire_year_extrapolation:外挿列(0.5以上): ['入社年']


## 11b. 特徴量グループの棚卸し（`40_` から移植）

In [48]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。
#   ベースラインを 40_ の構成に切り替えられるようにするために必要。

# ============================================================
# 特徴量グループの棚卸し
#   prepare_split() が merge している元フレームごとに列を分類する。
#   「どのグループにも属さない列」「2グループに重複する列」が出たら
#   減量の定義がずれているのでassertで止める。
# ============================================================

ALL_FEATS = set(_feature_cols(ag_train_80))

# prepare_split() 内で生成される派生列（元フレームを持たないのでここに明示）
DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
# 部署Target Encoding が生む列（create_department_target_encoding の出力）
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

# 実際に特徴量として残っている列だけに絞る（drop_colsで消えたものを自動的に除外）
FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


In [49]:
# ※ このセルは 40_feature_reduction.ipynb から移植したもの（内容は同一）。

# ============================================================
# 月次集約(agg)の「指標 × 統計」分解
#   create_monthly_aggregation_features が作る 16指標 × 14統計 を分解し、
#   冗長な統計を落とせるようにする。
# ============================================================

AGG_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
AGG_ALL_STATS = [
    "mean", "std", "min", "max", "median", "cv",
    "early_mean", "mid_mean", "late_mean", "late_minus_early", "late_early_ratio",
    "slope", "diff", "ratio",
]

# 残す統計。冗長性を根拠に選ぶ（検証スコアで選んでいない）:
#   median←mean と重複 / cv←std/mean の比 / min,max←外れ値1点 /
#   mid_mean←early,lateから内挿可能 / late_minus_early,late_early_ratio,diff,ratio←slopeと同義
AGG_KEEP_STATS = {"mean", "std", "early_mean", "late_mean", "slope"}


def _agg_stat(col):
    """agg列名を (指標, 統計) に分解して統計名を返す。最長一致で指標を特定する。"""
    best = None
    for m in AGG_METRICS:
        if col.startswith(m + "_") and (best is None or len(m) > len(best)):
            best = m
    if best is None:
        return None
    return col[len(best) + 1:]


_unmapped = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) not in AGG_ALL_STATS]
assert not _unmapped, f"指標×統計に分解できないagg列: {_unmapped}"

AGG_SLIM_COLS = [c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in AGG_KEEP_STATS]
print(f"agg: {len(FEATURE_GROUPS['agg'])} 列 → 統計を{sorted(AGG_KEEP_STATS)}に限定すると {len(AGG_SLIM_COLS)} 列")
print(f"落とす統計: {sorted(set(AGG_ALL_STATS) - AGG_KEEP_STATS)}")


agg: 224 列 → 統計を['early_mean', 'late_mean', 'mean', 'slope', 'std']に限定すると 80 列
落とす統計: ['cv', 'diff', 'late_early_ratio', 'late_minus_early', 'max', 'median', 'mid_mean', 'min', 'ratio']


## 11c. ベースライン構成の選択

**`40_` の Public 結果が出たら、次のセルの `BASE` だけを変える。**

In [50]:
# ============================================================
# ベースライン構成の選択
#   ★ 40_ の Public 結果が出たら、ここの BASE だけを変える ★
#
#   41_ が問うのは「入社年を外すべきか」であり、この問い自体はベースラインに依存しない。
#   ベースラインを切り替えるのは、提出が現時点の最良構成の上に載るようにするため。
# ============================================================

CORE_GROUPS = {"persona", "agg", "deptte", "derived", "L2"}

BASE_SPECS = {
    # 37_ D3（Public 0.522659）と同一。40_ の R0_ref が相関1.000000で再現済み
    "full441":     {"groups": ALL_GROUPS,              "agg_stats": None},
    # 40_ の R1_no_tfidf と同一（396列）
    "no_tfidf396": {"groups": ALL_GROUPS - {"tfidf"},  "agg_stats": None},
    # 40_ の R6_lean と同一（113列）
    "lean113":     {"groups": CORE_GROUPS,             "agg_stats": AGG_KEEP_STATS},
}

# ----------------------------------------------------------------
BASE = "lean113"     # 40_ R6_lean が Public 0.521729 で新最良（2026-08-12）。ここに合わせた
# ----------------------------------------------------------------

assert BASE in BASE_SPECS, f"BASEは {list(BASE_SPECS)} のいずれか"
BASE_SPEC = BASE_SPECS[BASE]


def _cols_of_spec(spec, df):
    """グループ指定から特徴量列を作る（元データフレームの列順を保つ）"""
    keep = set()
    for g in spec["groups"]:
        if g == "agg" and spec["agg_stats"] is not None:
            keep |= {c for c in FEATURE_GROUPS["agg"] if _agg_stat(c) in spec["agg_stats"]}
        else:
            keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


BASE_COLS = _cols_of_spec(BASE_SPEC, ag_full)
assert _cols_of_spec(BASE_SPEC, ag_train_80b) == BASE_COLS, \
    "80%学習と全件学習でベースラインの列が食い違っている"
assert "入社年" in BASE_COLS, \
    "ベースラインに入社年が含まれていない。41_の問いが成立しないので構成を見直すこと"

_expected = {"full441": 441, "no_tfidf396": 396, "lean113": 113}
print(f"ベースライン: {BASE}  →  {len(BASE_COLS)} 列")
if BASE in _expected:
    if len(BASE_COLS) == _expected[BASE]:
        print(f"  ✅ 40_ の実行結果（{_expected[BASE]}列）と一致")
    else:
        print(f"  ⚠️ 40_ の実行結果は {_expected[BASE]}列だった。パイプラインに差分がないか確認すること")
print(f"  除外グループ: {sorted(ALL_GROUPS - BASE_SPEC['groups']) or '（なし）'}"
      + ("  + agg統計を5種に限定" if BASE_SPEC["agg_stats"] is not None else ""))


ベースライン: lean113  →  113 列
  ✅ 40_ の実行結果（113列）と一致
  除外グループ: ['advstats', 'catchange', 'cluster', 'domain', 'edafeat', 'mgr', 'missing', 'quarterly', 'tfidf']  + agg統計を5種に限定


## 12. 構成の事前登録

In [51]:
# ============================================================
# 第12節: 構成の事前登録
#   すべての構成は BASE_COLS（前セルで選んだベースライン）から
#   列を引くだけで作る。したがって X0 は常に「そのベースラインそのもの」になる。
# ============================================================

A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_HOLDOUT = 560   # 38_ で 80%学習(2208件)での最適点と実測。350〜900は平坦
ITER_FULL    = 560   # D3(Public 0.522659)と同一。ここを変えると「特徴量だけの差」でなくなる

SEEDS_SUB = [42, 2024, 7, 1234, 99]                      # 提出用。D3・40_と同一の5シード
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]    # 検証用。8シード（40_と同一）

HIRE_TIME_COLS = ["入社年", "入社月", "入社四半期"]

# 外挿列のうち、実際にベースラインに含まれているものだけが除外対象になる
_extrap_in_base = [c for c in EXTRAPOLATING_COLS if c in BASE_COLS]

CONFIGS = {
    "X0_ref":             {"drop": []},
    "X1_no_hireyear":     {"drop": ["入社年"]},
    "X2_no_all_hiretime": {"drop": [c for c in HIRE_TIME_COLS if c in BASE_COLS]},
    "X3_no_extrapolating": {"drop": _extrap_in_base},
}

# X3 が X1 や X2 と同一なら冗長なので落とす
if set(CONFIGS["X3_no_extrapolating"]["drop"]) == set(CONFIGS["X1_no_hireyear"]["drop"]):
    del CONFIGS["X3_no_extrapolating"]
    print("外挿列は 入社年 のみだったため X3 は X1 と同一。X3をスキップする。")
elif set(CONFIGS["X3_no_extrapolating"]["drop"]) == set(CONFIGS["X2_no_all_hiretime"]["drop"]):
    del CONFIGS["X3_no_extrapolating"]
    print("外挿列が X2 と同一だったため X3 をスキップする。")

VAL_REJECT_MARGIN = 0.02   # X0からこれ以上悪化した構成は提出しない（事前登録）


def cols_for(spec, df):
    """ベースラインから spec["drop"] の列を引いて返す。

    BASE_COLS は既に df の列順で作られているので、順序はそのまま保たれる。
    列順を保つのは、BASE="full441" のとき X0_ref を D3 と同じ入力にするため。
    """
    drop = set(spec["drop"])
    base = _cols_of_spec(BASE_SPEC, df)
    return [c for c in base if c not in drop]


print()
print(f"{'config':<20s} {'列数':>5s}  除外する列")
print("-" * 70)
for _name, _spec in CONFIGS.items():
    print(f"{_name:<20s} {len(cols_for(_spec, ag_train_80b)):>5d}  "
          f"{', '.join(_spec['drop']) if _spec['drop'] else '（なし）'}")

assert cols_for(CONFIGS["X0_ref"], ag_full) == BASE_COLS, "X0がベースラインと一致しない"
assert len(cols_for(CONFIGS["X1_no_hireyear"], ag_full)) == len(BASE_COLS) - 1, \
    "X1がちょうど1列だけ少なくなっていない"
assert "入社年" not in cols_for(CONFIGS["X1_no_hireyear"], ag_full), "X1に入社年が残っている"
assert "入社月" in cols_for(CONFIGS["X1_no_hireyear"], ag_full) or "入社月" not in BASE_COLS, \
    "X1で入社月まで落ちている"
print()
print(f"✅ X0={len(BASE_COLS)}列 / X1={len(BASE_COLS) - 1}列（差は入社年の1列だけ）を確認")


外挿列は 入社年 のみだったため X3 は X1 と同一。X3をスキップする。

config                  列数  除外する列
----------------------------------------------------------------------
X0_ref                 113  （なし）
X1_no_hireyear         112  入社年
X2_no_all_hiretime     110  入社年, 入社月, 入社四半期

✅ X0=113列 / X1=112列（差は入社年の1列だけ）を確認


## 13. モデル関数

In [52]:
# ============================================================
# 第13節: モデル関数（反復数固定・特徴量列を明示的に受け取る）
# ============================================================

def _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed):
    model = cb.CatBoostClassifier(
        **params, iterations=int(n_iter), random_seed=seed,
        verbose=False, cat_features=obj_cols, task_type="CPU",
    )
    model.fit(X_tr, y_tr)
    return model


def fit_holdout_fixed(ag_train, ag_val, feature_cols, params, n_iter, seeds):
    """80/20ホールドアウトを反復数固定で学習。early stoppingは使わない
    （38_ で best_iteration が固定反復の最適点を系統的に下回ると分かったため）"""
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    val_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        val_preds.append(model.predict_proba(X_va)[:, 1])
    val_preds = np.array(val_preds)
    singles = [log_loss(y_va, vp) for vp in val_preds]
    return {
        "val_seedavg": float(log_loss(y_va, val_preds.mean(axis=0))),
        "val_single_mean": float(np.mean(singles)),
        "val_single_sd": float(np.std(singles)),
        "val_preds": val_preds,
        "y_val": y_va.values,
    }


def fit_full_fixed(ag_full_, test_feats, feature_cols, params, n_iter, seeds):
    """Train全件で学習して Test を予測"""
    obj_cols = [c for c in feature_cols if ag_full_[c].dtype == "object"]
    X_tr, y_tr = ag_full_[feature_cols].fillna(-999), ag_full_[TARGET_COL]
    X_test = test_feats[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(test_preds)


print("✅ モデル関数定義完了")


✅ モデル関数定義完了


In [53]:
# ============================================================
# チェックポイント（41_用にスキーマを差し替える）
#   make_row / load_checkpoint / save_checkpoint_row はグローバルの
#   RESULT_SCHEMA を参照するので、ここで上書きすれば流用できる。
# ============================================================

RESULT_SCHEMA = [
    "config", "n_features", "dropped_cols",
    "val_seedavg", "val_single_mean", "val_single_sd",
    "n_iterations", "n_train", "pred_mean", "submission_path",
]


def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label] if len(checkpoint) else checkpoint
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_seedavg={row.get('val_seedavg')}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result


_probe = make_row(config="__schema_probe__", n_features=1)
assert list(_probe.keys()) == RESULT_SCHEMA, "make_rowが新スキーマを見ていない"

_rejected = False
try:
    make_row(config="x", val_score=0.5)   # 旧(37_)スキーマのキー。弾かれるはず
except AssertionError as _e:
    _rejected = "RESULT_SCHEMA" in str(_e)
assert _rejected, "旧スキーマのキーが素通りした。RESULT_SCHEMAの差し替えが効いていない"

print("✅ チェックポイントを41_スキーマに差し替え完了")
print(f"   {RESULT_SCHEMA}")


✅ チェックポイントを41_スキーマに差し替え完了
   ['config', 'n_features', 'dropped_cols', 'val_seedavg', 'val_single_mean', 'val_single_sd', 'n_iterations', 'n_train', 'pred_mean', 'submission_path']


## 14. 実行

In [54]:
# ============================================================
# 第14節: 実行
#   全構成で共通:
#     - ハイパーパラメータ A_PARAMS 固定
#     - 検証 : 先頭80%学習 / 生存者535名 / 反復560固定 / 8シード平均
#     - 提出 : Train全件学習 / 反復560固定 / 5シード平均（D3と同一シード）
# ============================================================

def make_runner(config_label, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        feats_full = cols_for(spec, ag_full)
        assert feats == feats_full, "80%学習と全件学習で特徴量列が食い違っている"

        logger.info("=" * 60)
        logger.info(f"[{config_label}] {len(feats)}列 / 除外: {spec['drop'] or 'なし'}")

        hold = fit_holdout_fixed(ag_train_80b, ag_val_surv, feats, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一シード {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy", hold["val_preds"])

        test_preds = fit_full_fixed(ag_full, test_features_full, feats, A_PARAMS, ITER_FULL, SEEDS_SUB)
        preds = test_preds.mean(axis=0)
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_testpreds.npy", test_preds)
        path = save_submission(test_features_full.index, preds, config_label)

        return make_row(
            config=config_label, n_features=len(feats),
            dropped_cols=",".join(spec["drop"]),
            val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
            val_single_sd=hold["val_single_sd"],
            n_iterations=ITER_FULL, n_train=len(ag_full),
            pred_mean=float(preds.mean()), submission_path=path,
        )
    return _run


results = {}
for _name, _spec in CONFIGS.items():
    results[_name] = run_or_resume(_name, make_runner(_name, _spec))

print()
print(f"{'config':<20s} {'列数':>5s} {'val(8シード平均)':>16s} {'単一sd':>9s} {'予測平均':>9s}")
print("-" * 66)
for _name, _r in results.items():
    print(f"{_name:<20s} {int(_r['n_features']):>5d} {float(_r['val_seedavg']):>16.6f} "
          f"{float(_r['val_single_sd']):>9.6f} {float(_r['pred_mean']):>9.4f}")


[2026-08-12 00:30:07] [INFO] ============================================================


INFO:41_hire_year_extrapolation:============================================================


[2026-08-12 00:30:07] [INFO] [X0_ref] 113列 / 除外: なし


INFO:41_hire_year_extrapolation:[X0_ref] 113列 / 除外: なし


[2026-08-12 00:30:31] [INFO]   検証(生存者535名): シード平均 0.514642 / 単一シード 0.517727 ± 0.005220


...

[2026-08-12 00:32:13] [INFO]     seed=99: 全件学習完了


INFO:41_hire_year_extrapolation:    seed=99: 全件学習完了


[2026-08-12 00:32:13] [INFO]   提出ファイル: 20260812_41_hire_year_extrapolation_X2_no_all_hiretime.csv（予測平均=0.5877）


INFO:41_hire_year_extrapolation:  提出ファイル: 20260812_41_hire_year_extrapolation_X2_no_all_hiretime.csv（予測平均=0.5877）



config                  列数      val(8シード平均)      単一sd      予測平均
------------------------------------------------------------------
X0_ref                 113         0.514642  0.005220    0.5874
X1_no_hireyear         112         0.516969  0.004552    0.5879
X2_no_all_hiretime     110         0.513076  0.005974    0.5877


In [55]:
# ============================================================
# X0_ref の再現性チェック
#   X0 は D3(Public 0.522659) と同じ特徴量・パラメータ・反復数・シードなので
#   Test予測はほぼ一致するはず。ここがズレていたら以降の比較は無効。
# ============================================================

# D3 と比較できるのは BASE="full441" のときだけ。
# 減量済みベースラインに切り替えている場合、X0 は D3 と別物なので比較は無意味。
if BASE != "full441":
    print(f'BASE="{BASE}" のため D3 との再現チェックはスキップする。')
    print("（このベースラインの Public は 40_ の対応する提出で既に判明しているはず）")
    _skip_repro = True
else:
    _skip_repro = False

_x0 = pd.read_csv(results["X0_ref"]["submission_path"], header=None, names=[ID_COL, "pred"])
_d3_files = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_37_full_train_seed_averaging_D3_Aparams_full_x125.csv"))

if _d3_files and not _skip_repro:
    _d3f = pd.read_csv(_d3_files[-1], header=None, names=[ID_COL, "pred"])
    _mm = _x0.merge(_d3f, on=ID_COL, suffixes=("_x0", "_d3"))
    assert len(_mm) == len(_x0), "社員IDが一致しない"
    _corr = _mm["pred_x0"].corr(_mm["pred_d3"])
    _mad = (_mm["pred_x0"] - _mm["pred_d3"]).abs().mean()
    print(f"D3ファイル: {_d3_files[-1].name}")
    print(f"  相関           : {_corr:.6f}")
    print(f"  平均絶対差     : {_mad:.6f}")
    print(f"  予測平均 X0/D3 : {_mm['pred_x0'].mean():.4f} / {_mm['pred_d3'].mean():.4f}")
    if _corr > 0.999 and _mad < 0.005:
        print("✅ D3を再現できている。以降の構成差は特徴量に起因すると解釈してよい")
    else:
        print("⚠️ D3を再現できていない。パイプラインに差分があるので原因を特定すること")
elif not _skip_repro:
    print("⚠️ D3の提出ファイルが見つからなかった（再現チェックをスキップ）")


BASE="lean113" のため D3 との再現チェックはスキップする。
（このベースラインの Public は 40_ の対応する提出で既に判明しているはず）


## 15. 入社年の寄与を分離する

In [56]:
# ============================================================
# 第15節: 入社年の寄与を分離する
#   X0 と X1 は「入社年の有無」だけが違い、シードも同一。
#   したがって両者のTest予測の差が、そのまま 入社年 の寄与である。
#   文献レビュー第3.1節で「分離できていない」と留保した点をここで解決する。
# ============================================================

_p0 = pd.read_csv(results["X0_ref"]["submission_path"],
                  header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]
_p1 = pd.read_csv(results["X1_no_hireyear"]["submission_path"],
                  header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"].loc[_p0.index]

_year = pd.to_datetime(test_persona.set_index(ID_COL).loc[_p0.index, "入社日"]).dt.year
_diff = _p0 - _p1

print("=" * 70)
print("入社年の寄与 = X0(あり) − X1(なし) のTest予測差")
print("=" * 70)
print(f"  相関           : {_p0.corr(_p1):.6f}")
print(f"  平均絶対差     : {_diff.abs().mean():.6f}")
print(f"  最大絶対差     : {_diff.abs().max():.6f}")
print(f"  予測平均       : X0 {_p0.mean():.4f} / X1 {_p1.mean():.4f}（差 {_diff.mean():+.4f}）")
print()
print("入社年ごとの寄与:")
_tab = pd.DataFrame({"入社年": _year, "X0": _p0, "X1": _p1, "差": _diff})
print(_tab.groupby("入社年").agg(
    n=("差", "size"), X0平均=("X0", "mean"), X1平均=("X1", "mean"),
    差の平均=("差", "mean"), 差の絶対値平均=("差", lambda s: s.abs().mean())).round(4).to_string())

print()
print("=" * 70)
print("解釈の目安")
print("=" * 70)
_mad01 = _diff.abs().mean()
if _mad01 < 0.002:
    print(f"  平均絶対差 {_mad01:.6f} は極めて小さい。")
    print("  → 入社年はモデルにほとんど使われておらず、設計上の不備ではあるが実害は無い。")
    print("     提出してもPublicは 0.522659 の近傍に戻るだけの可能性が高い。")
elif _mad01 < 0.01:
    print(f"  平均絶対差 {_mad01:.6f} は小さいが無視はできない。")
    print("  → Publicで1提出して決着させる価値がある。")
else:
    print(f"  平均絶対差 {_mad01:.6f} は大きい。入社年は予測を実質的に動かしている。")
    print("  → 外挿域での分岐が効いている可能性が高く、除外の効果が期待できる。")

# --- 参考: 入社年の特徴量重要度 ---
logger.info("入社年の特徴量重要度を測定中（全件学習・seed=42の1本）")
_feats_all = cols_for(CONFIGS["X0_ref"], ag_full)   # = BASE_COLS
_obj = [c for c in _feats_all if ag_full[c].dtype == "object"]
_mdl = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER_FULL, random_seed=42,
                             verbose=False, cat_features=_obj, task_type="CPU")
_mdl.fit(ag_full[_feats_all].fillna(-999), ag_full[TARGET_COL])
_imp = pd.Series(_mdl.get_feature_importance(), index=_feats_all).sort_values(ascending=False)

print()
print("特徴量重要度（PredictionValuesChange, seed=42）")
print(f"  入社年の重要度  : {_imp.get('入社年', float('nan')):.4f}")
print(f"  入社年の順位    : {list(_imp.index).index('入社年') + 1} / {len(_imp)} 列")
print(f"  重要度の合計に占める割合: {100 * _imp.get('入社年', 0) / _imp.sum():.3f}%")
print()
print("  参考: 上位10列")
print(_imp.head(10).round(4).to_string())
print()
print("  参考: 入社月・入社四半期")
for _c in ["入社月", "入社四半期"]:
    if _c in _imp.index:
        print(f"    {_c}: 重要度 {_imp[_c]:.4f}（順位 {list(_imp.index).index(_c) + 1}）")

_imp.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_feature_importance.csv", header=["importance"])


入社年の寄与 = X0(あり) − X1(なし) のTest予測差
  相関           : 0.997292
  平均絶対差     : 0.014378
  最大絶対差     : 0.123393
  予測平均       : X0 0.5874 / X1 0.5879（差 -0.0006）

入社年ごとの寄与:
        n    X0平均    X1平均    差の平均  差の絶対値平均
入社年                                       
2014  829  0.5830  0.5843 -0.0013   0.0136
2015  855  0.6079  0.6077  0.0002   0.0152
2016  809  0.5701  0.5709 -0.0008   0.0143
2017    9  0.5885  0.5824  0.0061   0.0118

解釈の目安
  平均絶対差 0.014378 は大きい。入社年は予測を実質的に動かしている。
  → 外挿域での分岐が効いている可能性が高く、除外の効果が期待できる。
[2026-08-12 00:32:13] [INFO] 入社年の特徴量重要度を測定中（全件学習・seed=42の1本）


INFO:41_hire_year_extrapolation:入社年の特徴量重要度を測定中（全件学習・seed=42の1本）



特徴量重要度（PredictionValuesChange, seed=42）
  入社年の重要度  : 0.1826
  入社年の順位    : 98 / 113 列
  重要度の合計に占める割合: 0.183%

  参考: 上位10列
専攻分野                       6.4656
初期職種                       6.4369
残業時間_mean_job_deviation    5.7898
転居x勤務地_ダブル悪条件_v2           4.8080
残業時間_mean                  4.5115
残業時間_late_mean             4.1996
転居x勤務地_状態_v2               3.9064
360度評価_親和度_std             1.6934
上司との面談実施回数_std             1.6025
360度評価_信頼度_late_mean       1.6019

  参考: 入社月・入社四半期
    入社月: 重要度 0.0891（順位 102）
    入社四半期: 重要度 0.0630（順位 105）


## 16. 結果まとめ

In [57]:
# ============================================================
# 第16節: 結果まとめと提出方針
# ============================================================

_ref_val = float(results["X0_ref"]["val_seedavg"])
_ref_pred = pd.read_csv(results["X0_ref"]["submission_path"],
                        header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]

rows = []
for name, r in results.items():
    p = pd.read_csv(r["submission_path"], header=None,
                    names=[ID_COL, "pred"]).set_index(ID_COL)["pred"].loc[_ref_pred.index]
    val = float(r["val_seedavg"])
    rows.append({
        "config": name,
        "列数": int(r["n_features"]),
        "除外列": r["dropped_cols"] if isinstance(r["dropped_cols"], str) else "",
        "val(生存者)": val,
        "X0との差": val - _ref_val,
        "X0との相関": float(np.corrcoef(p.values, _ref_pred.values)[0, 1]),
        "平均絶対差": float(np.abs(p.values - _ref_pred.values).mean()),
        "予測平均": float(p.mean()),
        "ファイル": Path(r["submission_path"]).name,
    })

summary = pd.DataFrame(rows)
summary["提出"] = np.where(summary["config"] == "X0_ref", "不要（提出済み・参照用）",
                    np.where(summary["X0との差"] > VAL_REJECT_MARGIN, "見送り（足切り）", "提出する"))

pd.set_option("display.width", 220)
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")

print()
print("=" * 70)
print("提出順（現最良からの変更が小さい順）")
print("=" * 70)
_order = ["X1_no_hireyear", "X3_no_extrapolating", "X2_no_all_hiretime"]
_i = 0
for _c in _order:
    _row = summary[(summary["config"] == _c) & (summary["提出"] == "提出する")]
    if len(_row) == 0:
        continue
    _i += 1
    _r = _row.iloc[0]
    print(f"{_i}. {_c:<20s} {_r['ファイル']}")
    print(f"     列数 {_r['列数']} / X0との相関 {_r['X0との相関']:.5f} "
          f"/ 平均絶対差 {_r['平均絶対差']:.5f} / 予測平均 {_r['予測平均']:.4f}")

print()
print("※ X0との相関が0.9999を超え平均絶対差が0.002未満の構成は、38_のH1/H2と同じく")
print("   提出しても 0.522659 の近傍に戻るだけで情報が得られない可能性が高い。")
print("   その場合は『設計は直したが得点は動かなかった』という結論をレポートに残せば足りる。")


            config  列数           除外列  val(生存者)     X0との差   X0との相関    平均絶対差     予測平均                                                       ファイル           提出
            X0_ref 113                0.514642  0.000000 1.000000 0.000000 0.587364             20260812_41_hire_year_extrapolation_X0_ref.csv 不要（提出済み・参照用）
    X1_no_hireyear 112           入社年  0.516969  0.002326 0.997292 0.014378 0.587937     20260812_41_hire_year_extrapolation_X1_no_hireyear.csv         提出する
X2_no_all_hiretime 110 入社年,入社月,入社四半期  0.513076 -0.001566 0.997451 0.014023 0.587672 20260812_41_hire_year_extrapolation_X2_no_all_hiretime.csv         提出する
[2026-08-12 00:32:17] [INFO] サマリを保存: 20260812_41_hire_year_extrapolation_summary.csv


INFO:41_hire_year_extrapolation:サマリを保存: 20260812_41_hire_year_extrapolation_summary.csv



提出順（現最良からの変更が小さい順）
1. X1_no_hireyear       20260812_41_hire_year_extrapolation_X1_no_hireyear.csv
     列数 112 / X0との相関 0.99729 / 平均絶対差 0.01438 / 予測平均 0.5879
2. X2_no_all_hiretime   20260812_41_hire_year_extrapolation_X2_no_all_hiretime.csv
     列数 110 / X0との相関 0.99745 / 平均絶対差 0.01402 / 予測平均 0.5877

※ X0との相関が0.9999を超え平均絶対差が0.002未満の構成は、38_のH1/H2と同じく
   提出しても 0.522659 の近傍に戻るだけで情報が得られない可能性が高い。
   その場合は『設計は直したが得点は動かなかった』という結論をレポートに残せば足りる。


## 17. 提出方針と結果の解釈

### 提出するファイル

第16節で「提出する」となったものを、**現最良からの変更が小さい順**に提出する。

| 順 | config | 何を検証するか |
|---|---|---|
| 1 | `X1_no_hireyear` | 学習範囲外に全Testが落ちる `入社年` を外すと Public は改善するか |
| 2 | `X3_no_extrapolating` | 同種の外挿列が他にもあった場合、まとめて外すとどうなるか（無ければスキップ） |
| 3 | `X2_no_all_hiretime` | 外挿問題を持たない `入社月`・`入社四半期` まで落とすと悪化するか（対照） |

`X0_ref` は `37_` D3 と同一内容なので**提出しない**（第14〜15節の参照用）。

### 結果の解釈ルール（事前登録）

- **Public が 0.522659 より改善** → 外挿列の除去が有効。`40_` の減量方針と合わせて次の構成を組む
- **Public が 0.5227 ± 0.001** → `入社年` は実害が無かった。
  **それでも X1 を今後の基準構成にする**（無信号かつ外挿する列を残す理由が無いため）
- **Public が悪化** → `入社年` は何らかの形で有効に使われていた。
  その場合は「なぜ無信号の列が効くのか」を先に説明できるまで、この方向は打ち切る
- `X2` が `X1` より明確に悪ければ、`入社月`・`入社四半期` には実signalがあることの証拠になる

### やらないこと

- **検証スコアで提出構成を選び直さない。**
  `入社年` の期待効果（0.000〜0.003）は検証セットの分解能 ±0.011 を大きく下回るため、
  ここでの検証スコアは**足切り以外に使えない**
- **ハイパーパラメータの再探索をしない。** 1列の増減で再探索すると効果が分離できなくなる
- **外部の景気指標（有効求人倍率など）のマージをしない。**
  入社年と1対1対応するので情報が増えず、外挿問題も解消しない
  （文献レビュー第3.1節の補足を参照）

### 本ノートブックの位置づけ

これは**得点を伸ばす施策ではなく、設計上の不備を潰す作業**である。
第15節で `入社年` の重要度と予測差が測れるので、
**実行後すぐに「効果の上限」がわかる**。上限が小さければ提出せずに終えてよく、
その場合も「441列の中に、無信号かつ全Testが外挿域に落ちる列が1つ混じっていた」という
事実は `data/output/submit_result_report.md` に負の結果として残す価値がある。

### 関連

- 文献レビュー: `literature/01_early_turnover_literature_review.md` 第3.1節
- 減量の本体: `src/40_feature_reduction.ipynb`（`入社年` は `persona` グループに含まれるため、
  40_ の `R2_core`・`R6_lean` にも残っている。40_ と 41_ は独立に評価できる）
